In [12]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("heart.csv")  # make sure file is in same folder

# Preview
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [13]:
# Q1 Load the Heart Disease dataset. Create a Series from the 'age' column with patient IDs as the index. Examine the age distribution among patients with and without heart disease using descriptive statistics.
import pandas as pd
import numpy as np

df = pd.read_csv("heart.csv")
df.index = ['P' + str(i) for i in range(1, len(df)+1)]
age_series = pd.Series(df['age'].values, index=df.index)
with_disease = df[df['target'] == 1]['age']
without_disease = df[df['target'] == 0]['age']
print(with_disease.describe())
print(without_disease.describe())

count    165.000000
mean      52.496970
std        9.550651
min       29.000000
25%       44.000000
50%       52.000000
75%       59.000000
max       76.000000
Name: age, dtype: float64
count    138.000000
mean      56.601449
std        7.962082
min       35.000000
25%       52.000000
50%       58.000000
75%       62.000000
max       77.000000
Name: age, dtype: float64


In [15]:
# Q2 Examine the relationship between 'cholesterol', 'thalach' (maximum heart rate), and disease outcome using ufuncs. Determine which numeric feature is most strongly associated with heart disease.
chol_mean = np.mean(df['chol'])
thalach_mean = np.mean(df['thalach'])
corr = df[['chol', 'thalach', 'trestbps', 'target']].corr()
print(corr)
print(corr['target'].drop('target').abs().idxmax())

              chol   thalach  trestbps    target
chol      1.000000 -0.009940  0.123174 -0.085239
thalach  -0.009940  1.000000 -0.046698  0.421741
trestbps  0.123174 -0.046698  1.000000 -0.144931
target   -0.085239  0.421741 -0.144931  1.000000
thalach


In [16]:
# Q3 Apply hierarchical indexing on 'sex' and 'cp' (chest pain type). Examine disease prevalence at each hierarchical level. Use xs() to extract gender-specific chest pain patterns.
df_multi = df.set_index(['sex', 'cp'])
prevalence = df_multi.groupby(level=['sex', 'cp'])['target'].mean()
print(prevalence)
male = df_multi.xs(1, level='sex')
female = df_multi.xs(0, level='sex')
print(male.groupby('cp')['target'].mean())
print(female.groupby('cp')['target'].mean())

sex  cp
0    0     0.461538
     1     0.888889
     2     0.971429
     3     1.000000
1    0     0.201923
     1     0.781250
     2     0.673077
     3     0.631579
Name: target, dtype: float64
cp
0    0.201923
1    0.781250
2    0.673077
3    0.631579
Name: target, dtype: float64
cp
0    0.461538
1    0.888889
2    0.971429
3    1.000000
Name: target, dtype: float64


In [17]:
# Q4 Create a 'Risk Score' combining normalized age, cholesterol, and blood pressure using ufunc transformations. Rank patients by risk and build a stratified report by gender and chest pain type.
df['age_norm'] = (df['age'] - df['age'].min()) / (df['age'].max() - df['age'].min())
df['chol_norm'] = (df['chol'] - df['chol'].min()) / (df['chol'].max() - df['chol'].min())
df['bp_norm'] = (df['trestbps'] - df['trestbps'].min()) / (df['trestbps'].max() - df['trestbps'].min())
df['Risk Score'] = df['age_norm'] + df['chol_norm'] + df['bp_norm']
df['Rank'] = df['Risk Score'].rank(ascending=False)
report = df.groupby(['sex', 'cp'])['Risk Score'].mean()
print(report)

sex  cp
0    0     1.332480
     1     1.085691
     2     1.169098
     3     1.494514
1    0     1.147826
     1     1.047969
     2     1.089453
     3     1.205388
Name: Risk Score, dtype: float64


In [25]:
# Q5 Examine the effect of dropping versus imputing missing values in the 'ca' (number of vessels) and 'thal' columns on the overall distribution of key clinical features.
print(df[['ca', 'thal']].isnull().sum())
drop_df = df.dropna(subset=['ca', 'thal'])
impute_df = df.copy()
impute_df['ca'] = impute_df['ca'].fillna(df['ca'].median())
impute_df['thal'] = impute_df['thal'].fillna(df['thal'].median())
print(df.describe())
print(drop_df.describe())
print(impute_df.describe())

ca      0
thal    0
dtype: int64
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.366337    0.683168    0.966997  131.623762  246.264026    0.148515   
std      9.082101    0.466011    1.032052   17.538143   51.830751    0.356198   
min     29.000000    0.000000    0.000000   94.000000  126.000000    0.000000   
25%     47.500000    0.000000    0.000000  120.000000  211.000000    0.000000   
50%     55.000000    1.000000    1.000000  130.000000  240.000000    0.000000   
75%     61.000000    1.000000    2.000000  140.000000  274.500000    0.000000   
max     77.000000    1.000000    3.000000  200.000000  564.000000    1.000000   

          restecg     thalach       exang     oldpeak       slope          ca  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean     0.528053  149.646865    0.326733    1.039604    1.399340    0.7293

In [19]:
# Q6 Using boolean indexing, filter high-risk patients (age above 55 and cholesterol above 250). Perform index-aligned subtraction against the full dataset mean to identify statistical deviations.
high_risk = df[(df['age'] > 55) & (df['chol'] > 250)]
mean_vals = df.mean(numeric_only=True)
deviation = high_risk.select_dtypes(include=np.number) - mean_vals
print(deviation)

            age       sex        cp   trestbps        chol       fbs  \
P5     2.633663 -0.683168 -0.966997 -11.623762  107.735974 -0.148515   
P7     1.633663 -0.683168  0.033003   8.376238   47.735974 -0.148515   
P15    3.633663 -0.683168  2.033003  18.376238   36.735974  0.851485   
P17    3.633663 -0.683168  1.033003 -11.623762   93.735974 -0.148515   
P26   16.633663 -0.683168  0.033003  28.376238   55.735974 -0.148515   
...         ...       ...       ...        ...         ...       ...   
P270   1.633663  0.316832 -0.966997  -1.623762   36.735974  0.851485   
P278   2.633663  0.316832  0.033003  -7.623762   14.735974 -0.148515   
P279   3.633663 -0.683168  0.033003   4.376238   72.735974  0.851485   
P289   2.633663  0.316832 -0.966997 -21.623762   88.735974 -0.148515   
P292   3.633663  0.316832 -0.966997 -17.623762   71.735974 -0.148515   

       restecg    thalach     exang   oldpeak    slope        ca      thal  \
P5    0.471947  13.353135  0.673267 -0.439604  0.60066 -0

In [21]:
# Q7 Design a clinical classification function using apply() to label patients as 'Low Risk', 'Medium Risk', or 'High Risk'. Build a pivot table of risk categories by gender and chest pain type.
def classify(row):
    if row['Risk Score'] > 2:
        return 'High Risk'
    elif row['Risk Score'] > 1:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['Risk Category'] = df.apply(classify, axis=1)
pivot = pd.pivot_table(df, values='Risk Score', index='Risk Category', columns=['sex', 'cp'], aggfunc='mean')
print(pivot)

sex                   0                                       1            \
cp                    0         1         2         3         0         1   
Risk Category                                                               
Low Risk       0.830689  0.732448  0.748408       NaN  0.801724  0.736658   
Medium Risk    1.406273  1.368285  1.388588  1.494514  1.288138  1.234755   

sex                                
cp                    2         3  
Risk Category                      
Low Risk       0.799280  0.787546  
Medium Risk    1.256522  1.449129  


In [22]:
# Q8 Examine how index alignment behaves when performing addition and subtraction operations between a disease-positive Series and a disease-negative Series. Document NaN behavior.
pos = df[df['target'] == 1]['chol']
neg = df[df['target'] == 0]['chol']
add = pos + neg
sub = pos - neg
print(add)
print(sub)

P1     NaN
P10    NaN
P100   NaN
P101   NaN
P102   NaN
        ..
P95    NaN
P96    NaN
P97    NaN
P98    NaN
P99    NaN
Name: chol, Length: 303, dtype: float64
P1     NaN
P10    NaN
P100   NaN
P101   NaN
P102   NaN
        ..
P95    NaN
P96    NaN
P97    NaN
P98    NaN
P99    NaN
Name: chol, Length: 303, dtype: float64


In [23]:
# Q9 Examine the null value distribution across all columns using isnull().sum(). Determine which features require the most attention before further analysis.
nulls = df.isnull().sum()
print(nulls.sort_values(ascending=False))

age              0
sex              0
cp               0
trestbps         0
chol             0
fbs              0
restecg          0
thalach          0
exang            0
oldpeak          0
slope            0
ca               0
thal             0
target           0
age_norm         0
chol_norm        0
bp_norm          0
Risk Score       0
Rank             0
Risk Category    0
dtype: int64


In [24]:
# Q10 Construct a comprehensive heart disease risk analysis report using hierarchical indexing, ufuncs, null handling, and cross-tabulation.
crosstab = pd.crosstab(df['Risk Category'], [df['sex'], df['cp']])
summary = df.groupby(['sex', 'cp']).agg({'age': 'mean', 'chol': 'mean', 'Risk Score': 'mean', 'target': 'mean'})
print(crosstab)
print(summary)

sex             0              1            
cp              0   1   2  3   0   1   2   3
Risk Category                               
Low Risk        5   8  12  0  30  12  19   7
Medium Risk    34  10  23  4  74  20  33  12
              age        chol  Risk Score    target
sex cp                                             
0   0   57.256410  267.538462    1.332480  0.461538
    1   51.944444  251.444444    1.085691  0.888889
    2   54.971429  261.057143    1.169098  0.971429
    3   63.250000  247.000000    1.494514  1.000000
1   0   55.105769  243.605769    1.147826  0.201923
    1   51.031250  241.031250    1.047969  0.781250
    2   52.538462  231.134615    1.089453  0.673077
    3   54.315789  235.052632    1.205388  0.631579
